In [8]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [9]:
# 폰트 설정
font_path = 'C:/Windows/Fonts/Malgunsl.ttf'
fm.fontManager.addfont(font_path)
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.family'] = font_prop.get_name()
plt.rcParams['axes.unicode_minus'] = False

In [10]:
# 데이터 로드
df = pd.read_csv('seoul_data.csv', encoding='utf-8')
print(f"데이터 로드 완료: {df.shape[0]}개 행정동, {df.shape[1]}개 컬럼")
print(f"컬럼: {df.columns.tolist()}")

데이터 로드 완료: 426개 행정동, 12개 컬럼
컬럼: ['행정동', '노인_의료복지시설수', '노인_주거복지시설수', '장애인_생산품판매시설수', '장애인_의료재활시설수', '장애인_지역사회재활시설수', '장애인_직업재활시설수', '노인_여가복지시설수', '노인_일자리지원기관수', '노인교실수', '경로당수', '총시설수']


In [11]:
# 복지지수 산출
weights = {
    '노인_의료복지시설수': 3,
    '노인_주거복지시설수': 3,
    '장애인_의료재활시설수': 3,
    '장애인_지역사회재활시설수': 2,
    '장애인_직업재활시설수': 2,
    '장애인_생산품판매시설수': 1,
    '노인_여가복지시설수': 2,
    '노인_일자리지원기관수': 2,
    '노인교실수': 1,
    '경로당수': 1,
}

df['복지지수'] = sum(df[col] * w for col, w in weights.items())

print(f"\n복지지수 통계:")
print(f"  - 최고: {df['복지지수'].max():.1f} ({df.loc[df['복지지수'].idxmax(), '행정동']})")
print(f"  - 최저: {df['복지지수'].min():.1f}")
print(f"  - 평균: {df['복지지수'].mean():.1f}")
print(f"  - 중앙값: {df['복지지수'].median():.1f}")


복지지수 통계:
  - 최고: 150.0 (신림동)
  - 최저: 0.0
  - 평균: 12.6
  - 중앙값: 10.0


## 색상

In [12]:
main_color = '#4a9eff'
accent_color = '#ff6b6b'
success_color = '#51cf66'
warn_color = '#ffd43b'
colors_palette = [
    '#4a9eff', '#ff6b6b', '#51cf66', '#ffd43b', '#da77f2',
    '#ff922b', '#20c997', '#74c0fc', '#f783ac', '#a9e34b'
]

## 대시보드 Figure 생성

In [13]:
fig = plt.figure(figsize=(20, 24))
fig.patch.set_facecolor('#0f1117')

# 제목
fig.text(0.5, 0.97,
         '서울시 행정동별 복지시설 취약지역 분석',
         fontsize=26, fontweight='bold',
         ha='center', va='top', color='white',
         fontproperties=font_prop)

fig.text(0.5, 0.945,
         '파이썬데이터분석 5조 | seoul_data.csv 기반 시각화',
         fontsize=14,
         ha='center', va='top', color='#888',
         fontproperties=font_prop)

Text(0.5, 0.945, '파이썬데이터분석 5조 | seoul_data.csv 기반 시각화')

## CHART 1: 복지지수 상위 15 막대차트

In [14]:
ax1 = fig.add_axes([0.05, 0.72, 0.55, 0.20])
ax1.set_facecolor('#1a1d2e')

# 데이터 준비: 상위 15개 행정동
top15 = df.nlargest(15, '복지지수')[['행정동', '복지지수']].reset_index(drop=True)

# 막대 그리기
bars = ax1.barh(top15['행정동'], top15['복지지수'],
                 color=main_color, edgecolor='none', height=0.7)

# 1위(최고값)를 빨간색으로 강조
bars[0].set_color(accent_color)

# 각 막대 끝에 수치 레이블 추가
for bar, val in zip(bars, top15['복지지수']):
    ax1.text(val + 1,
             bar.get_y() + bar.get_height() / 2,
             str(val),
             va='center', ha='left', color='white',
             fontsize=9, fontproperties=font_prop)

# 평균선 추가
ax1.axvline(top15['복지지수'].mean(), color=warn_color,
            linestyle='--', alpha=0.7, linewidth=1.5)
ax1.text(top15['복지지수'].mean() + 0.5, 14, '평균',
         color=warn_color, fontsize=8, fontproperties=font_prop)

# 제목 및 축 설정
ax1.set_title('복지지수 상위 15개 행정동 (막대형 차트)',
              color='white', fontsize=13, fontweight='bold',
              fontproperties=font_prop, pad=10)
ax1.set_xlabel('복지지수', color='#aaa', fontproperties=font_prop)
ax1.tick_params(colors='white', labelsize=9)
ax1.set_facecolor('#1a1d2e')

# 스타일 정리
for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.xaxis.label.set_color('#aaa')
ax1.yaxis.set_tick_params(labelcolor='white')
ax1.xaxis.set_tick_params(labelcolor='#aaa')
for lbl in ax1.get_yticklabels():
    lbl.set_font_properties(font_prop)
for lbl in ax1.get_xticklabels():
    lbl.set_font_properties(font_prop)

# y축 역순 (1위가 맨 위에 오도록)
ax1.invert_yaxis()


## CHART 2: 시설 유형별 분포 원형 차트

In [15]:
ax2 = fig.add_axes([0.63, 0.72, 0.34, 0.20])
ax2.set_facecolor('#1a1d2e')

# 시설 유형별 합계 계산
facility_cols = list(weights.keys())
facility_sums = df[facility_cols].sum()

# 한글 라벨 정의
labels_kor = {
    '노인_의료복지시설수': '노인\n의료복지',
    '노인_주거복지시설수': '노인\n주거복지',
    '장애인_생산품판매시설수': '장애인\n생산품판매',
    '장애인_의료재활시설수': '장애인\n의료재활',
    '장애인_지역사회재활시설수': '장애인\n지역사회재활',
    '장애인_직업재활시설수': '장애인\n직업재활',
    '노인_여가복지시설수': '노인\n여가복지',
    '노인_일자리지원기관수': '노인\n일자리',
    '노인교실수': '노인교실',
    '경로당수': '경로당',
}

# 상위 5개 + 나머지는 '기타'로 묶기
top5_cols = facility_sums.nlargest(5).index.tolist()
others_sum = facility_sums.drop(top5_cols).sum()
pie_vals = list(facility_sums[top5_cols]) + [others_sum]
pie_labels = [labels_kor[c] for c in top5_cols] + ['기타']

# 원형 차트 그리기
explode = [0.05] * len(pie_vals)  # 모든 조각을 5% 바깥으로 밀어내기
wedges, texts, autotexts = ax2.pie(
    pie_vals,
    labels=pie_labels,
    autopct='%.1f%%',  # 소수점 1자리로 퍼센트 표시
    colors=colors_palette,
    explode=explode,
    shadow=False,
    startangle=140,
    textprops={'color': 'white', 'fontsize': 8},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 1.5}
)

# 텍스트 한글 폰트 적용
for t in texts:
    t.set_font_properties(font_prop)
for at in autotexts:
    at.set_fontsize(8)

ax2.set_title('시설 유형별 분포 (원형 차트)',
              color='white', fontsize=13, fontweight='bold',
              fontproperties=font_prop, pad=10)

Text(0.5, 1.0, '시설 유형별 분포 (원형 차트)')

## CHART 3: 총시설수 히스토그램

In [16]:
ax3 = fig.add_axes([0.05, 0.48, 0.42, 0.18])
ax3.set_facecolor('#1a1d2e')

# 히스토그램 그리기 (20개 구간)
n, bins, patches = ax3.hist(df['총시설수'], bins=20,
                             color=main_color, edgecolor='#0f1117', linewidth=0.8)

# 하위 25% 기준선 (취약지역 기준)
threshold = df['총시설수'].quantile(0.25)
ax3.axvline(threshold, color=accent_color, linestyle='--', linewidth=2,
            label=f'하위 25% 기준: {threshold:.0f}개')

# 평균선
ax3.axvline(df['총시설수'].mean(), color=warn_color, linestyle='--', linewidth=2,
            label=f'평균: {df["총시설수"].mean():.1f}개')

# 제목 및 축 설정
ax3.set_title('행정동별 총시설수 분포 (히스토그램)',
              color='white', fontsize=13, fontweight='bold',
              fontproperties=font_prop, pad=8)
ax3.set_xlabel('총시설수', color='#aaa', fontproperties=font_prop)
ax3.set_ylabel('행정동 수', color='#aaa', fontproperties=font_prop)

# 스타일 정리
for spine in ax3.spines.values():
    spine.set_color('#333')
ax3.tick_params(colors='#aaa')
for lbl in ax3.get_xticklabels():
    lbl.set_font_properties(font_prop)
for lbl in ax3.get_yticklabels():
    lbl.set_font_properties(font_prop)

# 범례 추가
leg = ax3.legend(fontsize=9, facecolor='#1a1d2e', edgecolor='#333', labelcolor='white')
for text in leg.get_texts():
    text.set_font_properties(font_prop)

## CHART 4: 노인 vs 장애인 시설 산포도

In [17]:
ax4 = fig.add_axes([0.55, 0.48, 0.42, 0.18])
ax4.set_facecolor('#1a1d2e')

# 노인시설 합계 계산 (6개 컬럼)
노인시설 = df[[
    '노인_의료복지시설수', '노인_주거복지시설수',
    '노인_여가복지시설수', '노인_일자리지원기관수',
    '노인교실수', '경로당수'
]].sum(axis=1)

# 장애인시설 합계 계산 (4개 컬럼)
장애인시설 = df[[
    '장애인_생산품판매시설수', '장애인_의료재활시설수',
    '장애인_지역사회재활시설수', '장애인_직업재활시설수'
]].sum(axis=1)

# 산포도 그리기 (복지지수에 따라 색상 변화)
scatter = ax4.scatter(노인시설, 장애인시설,
                      c=df['복지지수'],  # 복지지수로 색칠
                      cmap='YlOrRd',     # 노랑→주황→빨강 그라디언트
                      s=30, alpha=0.7, edgecolors='none')

# 상위 3개 행정동에 레이블 추가 (annotate)
top3 = df.nlargest(3, '복지지수')
for _, row in top3.iterrows():
    n_sum = row[[
        '노인_의료복지시설수', '노인_주거복지시설수',
        '노인_여가복지시설수', '노인_일자리지원기관수',
        '노인교실수', '경로당수'
    ]].sum()
    d_sum = row[[
        '장애인_생산품판매시설수', '장애인_의료재활시설수',
        '장애인_지역사회재활시설수', '장애인_직업재활시설수'
    ]].sum()
    ax4.annotate(row['행정동'],
                 xy=(n_sum, d_sum),        # 화살표 끝 (데이터 좌표)
                 xytext=(5, 5),            # 텍스트 위치 (offset points)
                 textcoords='offset points',
                 color='white', fontsize=8,
                 fontproperties=font_prop,
                 arrowprops=dict(arrowstyle='->', color='#aaa', lw=0.8))

# 컬러바 추가
cb = plt.colorbar(scatter, ax=ax4)
cb.set_label('복지지수', color='white', fontproperties=font_prop, fontsize=9)
cb.ax.yaxis.set_tick_params(color='white')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

# 제목 및 축 설정
ax4.set_title('노인시설 vs 장애인시설 산포도',
              color='white', fontsize=13, fontweight='bold',
              fontproperties=font_prop, pad=8)
ax4.set_xlabel('노인시설 합계', color='#aaa', fontproperties=font_prop)
ax4.set_ylabel('장애인시설 합계', color='#aaa', fontproperties=font_prop)

# 스타일 정리
for spine in ax4.spines.values():
    spine.set_color('#333')
ax4.tick_params(colors='#aaa')
for lbl in ax4.get_xticklabels():
    lbl.set_font_properties(font_prop)
for lbl in ax4.get_yticklabels():
    lbl.set_font_properties(font_prop)

## CHART 5: 취약지역 하위 15 막대차트

In [18]:
ax5 = fig.add_axes([0.05, 0.26, 0.55, 0.18])
ax5.set_facecolor('#1a1d2e')

# 시설 0개 초과인 동만 필터링해서 하위 15개 추출
bottom15 = df[df['총시설수'] > 0].nsmallest(15, '복지지수')[
    ['행정동', '복지지수']
].reset_index(drop=True)

# 막대 그리기 (빨간색으로 취약함을 표현)
bars5 = ax5.barh(bottom15['행정동'], bottom15['복지지수'],
                  color=accent_color, edgecolor='none', height=0.7)

# 각 막대 끝에 수치 표시
for bar, val in zip(bars5, bottom15['복지지수']):
    ax5.text(val + 0.1,
             bar.get_y() + bar.get_height() / 2,
             str(val),
             va='center', ha='left', color='white',
             fontsize=9, fontproperties=font_prop)

# 제목 및 축 설정
ax5.set_title('복지 취약지역 하위 15개 행정동 (복지지수 기준)',
              color='white', fontsize=13, fontweight='bold',
              fontproperties=font_prop, pad=8)
ax5.set_xlabel('복지지수', color='#aaa', fontproperties=font_prop)
ax5.tick_params(colors='white', labelsize=9)

# 스타일 정리
for spine in ax5.spines.values():
    spine.set_visible(False)
for lbl in ax5.get_yticklabels():
    lbl.set_font_properties(font_prop)
for lbl in ax5.get_xticklabels():
    lbl.set_font_properties(font_prop)

# y축 역순
ax5.invert_yaxis()

## CHART 6: 시설 간 상관관계 히트맵

In [19]:
ax6 = fig.add_axes([0.63, 0.26, 0.34, 0.18])
ax6.set_facecolor('#1a1d2e')

# 상관계수 행렬 계산
corr_cols = [
    '노인_의료복지시설수', '노인_주거복지시설수',
    '장애인_의료재활시설수', '장애인_지역사회재활시설수',
    '노인_여가복지시설수', '경로당수', '총시설수'
]
corr_labels = ['노인의료', '노인주거', '장애의료', '장애지역', '노인여가', '경로당', '총시설수']
corr_matrix = df[corr_cols].corr()

# 대각선 위쪽(상삼각) 마스킹 (중복 제거)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# 히트맵 그리기
sns.heatmap(corr_matrix,
            ax=ax6,
            mask=mask,
            annot=True,       # 각 셀에 수치 표시
            fmt='.2f',        # 소수점 2자리
            cmap='coolwarm',  # 파랑(음)↔빨강(양) 그라디언트
            center=0,         # 0을 중심으로 색상 설정
            xticklabels=corr_labels,
            yticklabels=corr_labels,
            annot_kws={'size': 8},
            linewidths=0.5,
            linecolor='#0f1117',
            cbar_kws={'shrink': 0.8})

# 제목
ax6.set_title('시설 간 상관관계 히트맵',
              color='white', fontsize=13, fontweight='bold',
              fontproperties=font_prop, pad=8)

# 스타일 정리
ax6.tick_params(colors='white', labelsize=8)
for lbl in ax6.get_xticklabels():
    lbl.set_font_properties(font_prop)
for lbl in ax6.get_yticklabels():
    lbl.set_font_properties(font_prop)
ax6.set_facecolor('#1a1d2e')

## CHART 7: 기술통계 요약 카드

In [20]:
ax7 = fig.add_axes([0.05, 0.10, 0.90, 0.13])
ax7.set_facecolor('#1a1d2e')
ax7.axis('off')

# 통계치 계산
mean_idx = df['복지지수'].mean()
vulnerable = (df['복지지수'] < mean_idx).sum()
good = (df['복지지수'] >= mean_idx).sum()
zero_facility = (df['총시설수'] == 0).sum()
max_row = df.loc[df['복지지수'].idxmax()]
min_nonzero = df[df['총시설수'] > 0].loc[df[df['총시설수'] > 0]['복지지수'].idxmin()]

# 카드 데이터
stats = [
    ('전체 행정동 수', f'{len(df)}개', main_color),
    ('시설 0개 취약지역', f'{zero_facility}개', accent_color),
    ('평균 이하 지역', f'{vulnerable}개\n({vulnerable/len(df)*100:.1f}%)', warn_color),
    ('복지지수 최고', f'{max_row["행정동"]}\n({int(max_row["복지지수"])}점)', success_color),
    ('평균 복지지수', f'{mean_idx:.1f}점', main_color),
    ('총시설 평균', f'{df["총시설수"].mean():.1f}개', '#da77f2'),
]

# 각 카드 그리기
box_w = 0.148
for i, (label, value, color) in enumerate(stats):
    x_start = 0.01 + i * (box_w + 0.01)
    
    # 카드 배경 사각형
    rect = plt.Rectangle((x_start, 0.05), box_w, 0.85,
                         transform=ax7.transAxes,
                         facecolor='#252840', edgecolor=color,
                         linewidth=2, clip_on=False)
    ax7.add_patch(rect)
    
    # 라벨 텍스트
    ax7.text(x_start + box_w / 2, 0.72,
             label,
             transform=ax7.transAxes,
             ha='center', va='center',
             color='#aaa', fontsize=10,
             fontproperties=font_prop)
    
    # 값 텍스트 (강조)
    ax7.text(x_start + box_w / 2, 0.32,
             value,
             transform=ax7.transAxes,
             ha='center', va='center',
             color=color, fontsize=13, fontweight='bold',
             fontproperties=font_prop)

# 카드 그룹 제목
ax7.set_title('기술통계 요약',
              color='white', fontsize=13, fontweight='bold',
              fontproperties=font_prop, pad=8, loc='left', x=0.01)

Text(0.01, 1.0, '기술통계 요약')

In [21]:
# 푸터
fig.text(0.5, 0.03,
         '데이터 출처: 공공데이터포털 | 분석 방법: 시각화 - 막대차트, 원형차트, 히스토그램, 산포도, 히트맵 | 파이썬데이터분석 5조',
         ha='center', color='#555', fontsize=9, fontproperties=font_prop)

Text(0.5, 0.03, '데이터 출처: 공공데이터포털 | 분석 방법: 시각화 - 막대차트, 원형차트, 히스토그램, 산포도, 히트맵 | 파이썬데이터분석 5조')

In [22]:
# 저장
plt.savefig('seoul_welfare_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117', edgecolor='none')
print("\n✓ 대시보드 저장 완료: seoul_welfare_analysis.png")

# 메모리 정리
plt.close()


✓ 대시보드 저장 완료: seoul_welfare_analysis.png


## 분석 결과 요약 출력

In [24]:
print("\n" + "="*60)
print("분석 결과 요약")
print("="*60)

print(f"\n📊 1. 복지지수 상위 15개 행정동:")
top15_detail = df.nlargest(15, '복지지수')[['행정동', '복지지수', '총시설수']]
for idx, (_, row) in enumerate(top15_detail.iterrows(), 1):
    print(f"   {idx:2d}. {row['행정동']:10s} - 복지지수: {row['복지지수']:3.0f}, 시설수: {int(row['총시설수']):3d}개")

print(f"\n📊 2. 시설 유형별 합계:")
for col, weight in sorted(weights.items(), key=lambda x: -x[1]):
    total = df[col].sum()
    print(f"   {col:20s}: {total:6.0f}개 (가중치: {weight})")

print(f"\n📊 3. 취약 지역 분석:")
print(f"   - 시설 0개 행정동: {zero_facility}개 ({zero_facility/len(df)*100:.1f}%)")
print(f"   - 복지지수 평균 이하 행정동: {vulnerable}개 ({vulnerable/len(df)*100:.1f}%)")
print(f"   - 평균 복지지수: {mean_idx:.2f}점")
print(f"   - 중앙값 복지지수: {df['복지지수'].median():.2f}점")

print(f"\n📊 4. 노인 vs 장애인 시설 비율:")
total_elderly = df[[col for col in df.columns if col.startswith('노인')]].sum().sum()
total_disabled = df[[col for col in df.columns if col.startswith('장애')]].sum().sum()
print(f"   - 노인시설 합계: {total_elderly:,.0f}개 ({total_elderly/(total_elderly+total_disabled)*100:.1f}%)")
print(f"   - 장애인시설 합계: {total_disabled:,.0f}개 ({total_disabled/(total_elderly+total_disabled)*100:.1f}%)")



분석 결과 요약

📊 1. 복지지수 상위 15개 행정동:
    1. 신림동        - 복지지수: 150, 시설수:  92개
    2. 미아동        - 복지지수:  70, 시설수:  56개
    3. 독산1동       - 복지지수:  62, 시설수:  39개
    4. 제기동        - 복지지수:  56, 시설수:  29개
    5. 상계1동       - 복지지수:  49, 시설수:  37개
    6. 진관동        - 복지지수:  49, 시설수:  43개
    7. 신사동        - 복지지수:  49, 시설수:  38개
    8. 신사동        - 복지지수:  49, 시설수:  38개
    9. 대림2동       - 복지지수:  43, 시설수:  19개
   10. 신당동        - 복지지수:  42, 시설수:  30개
   11. 공덕동        - 복지지수:  40, 시설수:  31개
   12. 상계2동       - 복지지수:  39, 시설수:  25개
   13. 세곡동        - 복지지수:  39, 시설수:  33개
   14. 강일동        - 복지지수:  39, 시설수:  26개
   15. 평창동        - 복지지수:  36, 시설수:  18개

📊 2. 시설 유형별 합계:
   노인_의료복지시설수          :    197개 (가중치: 3)
   노인_주거복지시설수          :     10개 (가중치: 3)
   장애인_의료재활시설수         :      2개 (가중치: 3)
   장애인_지역사회재활시설수       :    103개 (가중치: 2)
   장애인_직업재활시설수         :     55개 (가중치: 2)
   노인_여가복지시설수          :    240개 (가중치: 2)
   노인_일자리지원기관수         :     25개 (가중치: 2)
   장애인_생산품판매시설수        :      0개 (가중치: 1)